**Important:** These notebooks are sequential sections of the master notebook. Run them in order in the **same Colab runtime**; restarting the runtime clears required variables. The self-contained master notebook is recommended for a fresh run. The final-test section requires the locked-predictions GitHub commit before unlocking test labels.

## 2. Fold-local PyTorch MLP engine

In [ ]:
# Shared modeling engine — run in 02_nested_cv.ipynb
from dataclasses import dataclass
import copy
from sklearn.base import BaseEstimator,TransformerMixin
class ClipQuantiles(BaseEstimator,TransformerMixin):
 def __init__(self,enabled=False):self.enabled=enabled
 def fit(self,X,y=None):
  a=np.asarray(X,dtype=float)
  if self.enabled:self.lo_=np.nanpercentile(a,1,axis=0);self.hi_=np.nanpercentile(a,99,axis=0)
  return self
 def transform(self,X):
  a=np.asarray(X,dtype=float)
  return np.clip(a,self.lo_,self.hi_) if self.enabled else a

def preprocess(cfg,Xfit):
 nums=Xfit.select_dtypes(include=np.number).columns.tolist()
 cats=Xfit.select_dtypes(exclude=np.number).columns.tolist()
 scale={'Standard':'StandardScaler','MinMax':'MinMaxScaler','Robust':'RobustScaler'}
 scaler={'Standard':StandardScaler,'MinMax':MinMaxScaler,'Robust':RobustScaler}[cfg['scaler']]()
 numeric=Pipeline([('clip',ClipQuantiles(cfg.get('clip',False))),('impute',SimpleImputer(strategy='median')),('scale',scaler)])
 categorical=Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore'))])
 # pdays=999 is a meaningful sentinel; do not percentile-clip that column.
 special=['pdays'] if 'pdays' in nums else []
 regular=[c for c in nums if c!='pdays']
 transformers=[]
 if regular:transformers.append(('numeric',numeric,regular))
 if special:transformers.append(('pdays',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',{'Standard':StandardScaler,'MinMax':MinMaxScaler,'Robust':RobustScaler}[cfg['scaler']]())]),special))
 if cats:transformers.append(('categorical',categorical,cats))
 return ColumnTransformer(transformers,sparse_threshold=0)
class Net(nn.Module):
 def __init__(self,d,cfg):
  super().__init__(); layers=[]; act=nn.ReLU if cfg['activation']=='relu' else nn.Tanh
  for h in cfg['hidden']:
   layers.extend([nn.Linear(d,h),act(),nn.Dropout(cfg['dropout'])]);d=h
  layers.append(nn.Linear(d,1));self.net=nn.Sequential(*layers)
 def forward(self,x):return self.net(x).squeeze(-1)
def fit_model(Xtr,ytr,cfg,seed,tag):
 torch.manual_seed(seed);np.random.seed(seed);random.seed(seed)
 fit_ids,mon_ids=train_test_split(np.arange(len(ytr)),test_size=.15,stratify=ytr,random_state=seed)
 pp=preprocess(cfg,Xtr.iloc[fit_ids]);pp.fit(Xtr.iloc[fit_ids])
 a=np.asarray(pp.transform(Xtr.iloc[fit_ids]),dtype=np.float32)
 b=np.asarray(pp.transform(Xtr.iloc[mon_ids]),dtype=np.float32)
 ya=np.asarray(ytr.iloc[fit_ids],dtype=np.float32);yb=np.asarray(ytr.iloc[mon_ids],dtype=np.float32)
 net=Net(a.shape[1],cfg)
 optclass={'AdamW':torch.optim.AdamW,'Adam':torch.optim.Adam,'SGD':torch.optim.SGD}[cfg['optimizer']]
 kw={'lr':cfg['lr'],'weight_decay':cfg['wd']}
 if cfg['optimizer']=='SGD':kw['momentum']=.9
 opt=optclass(net.parameters(),**kw)
 scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=.5,patience=5,min_lr=1e-5) if cfg['scheduler'] else None
 lossfn=nn.BCEWithLogitsLoss();best=float('inf');wait=0;history=[];start=time.time()
 dataset=torch.utils.data.TensorDataset(torch.tensor(a),torch.tensor(ya))
 loader=torch.utils.data.DataLoader(dataset,batch_size=cfg['batch'],shuffle=True,generator=torch.Generator().manual_seed(seed))
 for epoch in range(1,151):
  net.train();losses=[]
  for xx,yy in loader:
   opt.zero_grad();loss=lossfn(net(xx),yy);loss.backward();opt.step();losses.append(loss.item())
  net.eval()
  with torch.no_grad(): val=lossfn(net(torch.tensor(b)),torch.tensor(yb)).item()
  lr=opt.param_groups[0]['lr'];history.append({'epoch':epoch,'train_loss':float(np.mean(losses)),'monitor_loss':val,'lr':lr})
  if val<best-1e-6:best=val;state=copy.deepcopy(net.state_dict());best_epoch=epoch;wait=0
  else:wait+=1
  if scheduler:scheduler.step(val)
  if wait>=15:break
 net.load_state_dict(state)
 pd.DataFrame(history).to_csv(ROOT/'results'/f'{tag}_history.csv',index=False)
 return pp,net,{'best_epoch':best_epoch,'elapsed_s':time.time()-start,'parameters':sum(t.numel() for t in net.parameters())}
def predict_prob(model,X):
 pp,net,_=model;net.eval()
 with torch.no_grad():return torch.sigmoid(net(torch.tensor(np.asarray(pp.transform(X),dtype=np.float32)))).numpy()
def metrics(y,p):
 pred=(p>=.5).astype(int);tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
 return dict(AP=average_precision_score(y,p),ROC_AUC=roc_auc_score(y,p),Accuracy=accuracy_score(y,pred),Precision=precision_score(y,pred,zero_division=0),Sensitivity=recall_score(y,pred,zero_division=0),Specificity=tn/(tn+fp),F1=f1_score(y,pred,zero_division=0),Balanced_Accuracy=balanced_accuracy_score(y,pred))
def candidate_eval(X,y,cfg,folds,tag,seed):
 scores=[];times=[];epochs=[]
 for j,(tr,va) in enumerate(folds):
  m=fit_model(X.iloc[tr].reset_index(drop=True),y.iloc[tr].reset_index(drop=True),cfg,seed+j,f'{tag}_inner{j}')
  scores.append(average_precision_score(y.iloc[va],predict_prob(m,X.iloc[va])));times.append(m[2]['elapsed_s']);epochs.append(m[2]['best_epoch'])
 return {'mean_AP':float(np.mean(scores)),'sd_AP':float(np.std(scores,ddof=1)),'time_s':float(sum(times)),'best_epoch_mean':float(np.mean(epochs))}


## 3. Freeze 6 manual + 6 random candidates

In [ ]:
base=dict(optimizer='AdamW',hidden=(32,),scaler='Standard',lr=.001,wd=1e-4,scheduler=True,batch=64,dropout=0.,activation='relu',clip=False)
manual={f'M{i}':dict(base) for i in range(1,7)}
manual['M2']['optimizer']='Adam';manual['M3']['optimizer']='SGD';manual['M4']['scaler']='MinMax';manual['M5']['hidden']=(64,32);manual['M6']['scheduler']=False
rng=np.random.default_rng(SEED)
random_cfg={}
for opt in ['SGD','Adam','AdamW']:
 for j in range(2):
  cfg=dict(optimizer=opt,hidden=tuple([(16,),(32,),(64,),(64,32),(128,64)][rng.integers(5)]),activation=str(rng.choice(['relu','tanh'])),lr=float(10**rng.uniform(-3,-1) if opt=='SGD' else 10**rng.uniform(-4,-2)),wd=float(rng.choice([0,1e-5,1e-4,1e-3])),batch=int(rng.choice([32,64,128])),scaler=str(rng.choice(['Standard','MinMax','Robust'])),dropout=float(rng.choice([0,.2])),clip=bool(rng.choice([False,True])),scheduler=True)
  random_cfg[f'R{len(random_cfg)+1}']=cfg
with open(ROOT/'configs/candidates.json','w') as f:json.dump({'manual':manual,'random':random_cfg,'seed':SEED},f,indent=2)
print('Candidates frozen in configs/candidates.json')


## 4. Outer 3-fold × inner 3-fold nested CV (long runtime)

In [ ]:
outer=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED+1)
outer_rows=[];search_rows=[]
for oi,(otr,ova) in enumerate(outer.split(Xdev,ydev),1):
 xx=Xdev.iloc[otr].reset_index(drop=True);yy=ydev.iloc[otr].reset_index(drop=True)
 inner=list(StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED+100+oi).split(xx,yy))
 for method,candidates in [('manual',manual),('random',random_cfg)]:
  scored=[]
  for name,cfg in candidates.items():
   result=candidate_eval(xx,yy,cfg,inner,f'outer{oi}_{name}_{name}',SEED+oi*1000)
   row={'outer_fold':oi,'method':method,'candidate':name,**result,**{k:str(v) for k,v in cfg.items()}}
   search_rows.append(row);scored.append((result['mean_AP'],name))
  winner=max(scored,key=lambda z:z[0])[1]
  chosen=candidates[winner]
  m=fit_model(xx,yy,chosen,SEED+oi,f'outer{oi}_{method}_refit')
  out=metrics(ydev.iloc[ova],predict_prob(m,Xdev.iloc[ova]))
  outer_rows.append({'outer_fold':oi,'method':method,'candidate':winner,**out,**m[2]})
 for name,model in [('Dummy',DummyClassifier(strategy='prior')),('LogisticRegression',LogisticRegression(max_iter=2000,random_state=SEED))]:
  pp=preprocess(base,xx);a=pp.fit_transform(xx);b=pp.transform(Xdev.iloc[ova]);model.fit(a,yy)
  pr=model.predict_proba(b)[:,1]
  outer_rows.append({'outer_fold':oi,'method':name,'candidate':'fixed',**metrics(ydev.iloc[ova],pr)})
 pd.DataFrame(search_rows).to_csv(ROOT/'results/search_all_folds.csv',index=False)
 pd.DataFrame(outer_rows).to_csv(ROOT/'results/outer_cv_results.csv',index=False)
 print('Outer fold completed:',oi)
res=pd.DataFrame(search_rows)
for method in ['manual','random']:
 res[res.method==method].to_csv(ROOT/'results'/f'{method}_search_results.csv',index=False)
outer_df=pd.DataFrame(outer_rows)
summary=outer_df.groupby('method')[['AP','ROC_AUC','F1','Sensitivity','Specificity','Balanced_Accuracy']].agg(['mean','std'])
display(summary);summary.to_csv(ROOT/'results/outer_cv_summary.csv')

# Comparison chart (outer evaluation AP, no tuning on outer scores)
fig,ax=plt.subplots(figsize=(7,4));outer_df.boxplot(column='AP',by='method',ax=ax);fig.suptitle('');ax.set_title('Outer-fold AP by method');fig.tight_layout();fig.savefig(ROOT/'figures/outer_method_AP.png');plt.show()


## 5. Fresh development-only 3-fold inner CV for final selection

In [ ]:
# Fresh inner CV over ALL development; do not select using outer results
folds=list(StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED+500).split(Xdev,ydev))
final_rows=[]
for method,candidates in [('manual',manual),('random',random_cfg)]:
 for name,cfg in candidates.items():
  r=candidate_eval(Xdev,ydev,cfg,folds,f'final_{name}',SEED+600)
  final_rows.append({'method':method,'candidate':name,**r})
final_df=pd.DataFrame(final_rows);final_df.to_csv(ROOT/'results/final_selection.csv',index=False)
w=final_df.sort_values('mean_AP',ascending=False).iloc[0]
selected=(manual if w.method=='manual' else random_cfg)[w.candidate]
with open(ROOT/'results/final_hyperparameters.json','w') as f:json.dump({'method':w.method,'candidate':w.candidate,'config':selected,'primary_seed':SEED},f,indent=2)
print('Selected:',w.method,w.candidate,'AP:',w.mean_AP)

fig,ax=plt.subplots(figsize=(9,4));final_df.set_index('candidate')['mean_AP'].plot.bar(ax=ax);ax.set_ylabel('Development inner-CV mean AP');fig.tight_layout();fig.savefig(ROOT/'figures/manual_random_comparison.png');plt.show()
